# H100 / B200 GPU Benchmark (Colab)

Identifies the GPU, measures dense matmul throughput in FP32, BF16 and FP8, and checks whether model weights fit in memory.

**Before running:** Runtime → Change runtime type → GPU. Record which GPU Colab assigned. Colab does not always give an H100, and it does not offer a B200 on standard tiers. The notebook reports what it gets and labels the result accordingly.

The results section at the bottom is for the numbers from your run only. Do not copy numbers from other runs into it.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,compute_cap --format=csv

In [ ]:
import torch, platform
from datetime import datetime, timezone

assert torch.cuda.is_available(), 'Switch the runtime to a GPU'

ARCH_BY_CC = {(8, 0): 'Ampere (A100)', (9, 0): 'Hopper (H100)', (10, 0): 'Blackwell (B200)'}
p = torch.cuda.get_device_properties(0)
info = {
    'name': p.name,
    'compute_capability': f'{p.major}.{p.minor}',
    'architecture': ARCH_BY_CC.get((p.major, p.minor), 'other'),
    'memory_gib': round(p.total_memory / 1024**3, 1),
    'sm_count': p.multi_processor_count,
    'bf16_supported': torch.cuda.is_bf16_supported(),
    'fp8_dtype_available': hasattr(torch, 'float8_e4m3fn'),
    'torch': torch.__version__,
    'cuda': torch.version.cuda,
    'python': platform.python_version(),
}
for k, v in info.items():
    print(f'{k:20s} {v}')
if info['architecture'] not in ('Hopper (H100)', 'Blackwell (B200)'):
    print('\nNote: this runtime is not H100 or B200. Results apply only to this GPU.')

In [ ]:
def time_per_iter(fn, warmup=5, iters=50):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(iters):
        fn()
    end.record()
    torch.cuda.synchronize()
    return start.elapsed_time(end) / 1000.0 / iters

def dense_tflops(n, dtype):
    a = torch.randn(n, n, device='cuda').to(dtype)
    b = torch.randn(n, n, device='cuda').to(dtype)
    return 2 * n**3 / time_per_iter(lambda: a @ b) / 1e12

def fp8_tflops(n):
    # _scaled_mm needs the second operand column-major, so transpose a contiguous tensor
    a = torch.randn(n, n, device='cuda').to(torch.float8_e4m3fn)
    b = torch.randn(n, n, device='cuda').to(torch.float8_e4m3fn).t()
    s = torch.tensor(1.0, device='cuda', dtype=torch.float32)
    fn = lambda: torch._scaled_mm(a, b, scale_a=s, scale_b=s, out_dtype=torch.bfloat16)
    return 2 * n**3 / time_per_iter(fn) / 1e12

sizes = [4096, 8192]
results = {}
for n in sizes:
    row = {'FP32': dense_tflops(n, torch.float32), 'BF16': dense_tflops(n, torch.bfloat16)}
    try:
        row['FP8'] = fp8_tflops(n)
    except Exception as exc:
        row['FP8'] = f'unavailable ({type(exc).__name__})'
    results[n] = row
    fp8 = row['FP8'] if isinstance(row['FP8'], str) else f"{row['FP8']:.1f}"
    print(f"n={n}: FP32 {row['FP32']:.1f} | BF16 {row['BF16']:.1f} | FP8 {fp8} TFLOP/s")

## Memory headroom for LLM weights
Weights only. Keeps 15% of free memory for KV cache and activations. Real serving needs a measured check.

In [ ]:
free, total = torch.cuda.mem_get_info()
free_gib = free / 1024**3
print(f'Free: {free_gib:.1f} GiB of {total / 1024**3:.1f} GiB')
for params_b in [8, 70]:
    for label, bpp in [('BF16', 2), ('FP8', 1)]:
        gib = params_b * 1e9 * bpp / 1024**3
        ok = gib <= free_gib * 0.85
        print(f'{params_b}B {label}: {gib:.1f} GiB -> {"fits" if ok else "does not fit"}')

## Results (fill in from your run)

| Field | Value |
|---|---|
| GPU name | *from the device cell* |
| Compute capability | |
| Driver / CUDA / torch | |
| FP32 / BF16 / FP8 at n=8192 (TFLOP/s) | |
| Date run (UTC) | |
| Runtime type | |

Limits: single GPU, one matmul shape, no multi-GPU or NCCL test, weights-only memory estimate.